In [257]:
import requests

In [258]:
url = "https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"

In [ ]:
response = requests.get(url)
print(response)

# Download books

In [166]:
import os

##output_path = f'./books/{filename}'

for i, line in enumerate(response.text.split(sep="\n")):
    if i == 0:
        continue
    print(line)
    book_url=line.split(sep=',')
   # print(len(book_url))
    if len(book_url) < 2:
        print("Skipping malformed line:", line)
        continue
    book_url_pdf = book_url[2]
    filename = os.path.basename(book_url_pdf)
    filename = filename.replace("\x00", "")

    response = requests.get(book_url_pdf)
    output_path = f'./books/{filename}'
    with open(output_path,"wb") as f:
        print( f'writing now...{output_path}')
        f.write(response.content) 
            


Think Python 2e,https://greenteapress.com/wp/think-python-2e/,https://greenteapress.com/thinkpython2/thinkpython2.pdf
writing now..../books/thinkpython2.pdf
Think DSP,https://greenteapress.com/wp/think-dsp/,https://greenteapress.com/thinkdsp/thinkdsp.pdf
writing now..../books/thinkdsp.pdf
Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,https://greenteapress.com/complexity2/thinkcomplexity2.pdf
writing now..../books/thinkcomplexity2.pdf
Think Java 2e,https://greenteapress.com/wp/think-java-2e/,https://greenteapress.com/thinkjava7/thinkjava2.pdf
writing now..../books/thinkjava2.pdf
Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling-in-matlab/,https://github.com/AllenDowney/PhysicalModelingInMatlab/raw/master/PhysicalModelingInMatlab4.pdf
writing now..../books/PhysicalModelingInMatlab4.pdf
Think OS,https://greenteapress.com/wp/think-os/,https://greenteapress.com/thinkos/thinkos.pdf
writing now..../books/thinkos.pdf
Think C++,https://greenteapress

# Convert to PDF

In [170]:
!uv add "markitdown[pdf]"


Resolved 141 packages in 2.23s
Prepared 13 packages in 2.69s
Installed 17 packages in 1.12s
 + click==8.5.0
 + coloredlogs==15.0.1
 + cryptography==50.0.2
 + flatbuffers==25.12.19
 + humanfriendly==10.0
 + magika==0.6.3
 + markdownify==1.2.3
 + markitdown==0.1.8
 + mpmath==1.3.0
 + onnxruntime==1.20.1
 + pdfminer-six==20260107
 + pdfplumber==0.11.10
 + pillow==12.3.0
 + protobuf==7.36.2
 + pypdfium2==5.13.0
 + pyreadline3==3.5.6
 + sympy==1.14.0


In [171]:
import os
from markitdown import MarkItDown

input_dir = "./books"        # folder containing PDFs
md = MarkItDown()

for filename in os.listdir(input_dir):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(input_dir, filename)

        # Output .md filename
        base_name = os.path.splitext(filename)[0]
        md_path = os.path.join(input_dir, base_name + ".md")

        print(f"Converting: {pdf_path}")

        result = md.convert(pdf_path)

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(result.text_content)

        print(f"Saved: {md_path}\n")


Converting: ./books\PhysicalModelingInMatlab4.pdf
Saved: ./books\PhysicalModelingInMatlab4.md

Converting: ./books\Think-C.pdf
Saved: ./books\Think-C.md

Converting: ./books\thinkcomplexity2.pdf
Saved: ./books\thinkcomplexity2.md

Converting: ./books\thinkdsp.pdf
Saved: ./books\thinkdsp.md

Converting: ./books\thinkjava2.pdf
Saved: ./books\thinkjava2.md

Converting: ./books\thinkos.pdf
Saved: ./books\thinkos.md

Converting: ./books\thinkpython2.pdf
Saved: ./books\thinkpython2.md



In [175]:
!wc -l ./books/thinkpython2.md

9702 ./books/thinkpython2.md


# Question 2 - Chunking for RAG

In [ ]:
import os

input_dir = "./books"

documents = []

for filename in os.listdir(input_dir):
    if filename.lower().endswith(".md"):
        path = os.path.join(input_dir, filename)

        with open(path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        # Remove empty / whitespace-only lines
        clean_lines = [line for line in lines if line.strip()]

        book_dict = {
            "source": filename,
            "content": clean_lines
        }

        documents.append(book_dict)

# Preview the first book
documents[0]



In [ ]:
from gitsource import chunk_documents


In [183]:
chunks = chunk_documents(documents,100,50)

In [191]:
source_name = "thinkpython2.md"

count = sum(1 for c in chunks if c["source"] == source_name)

print(count)


189


# Question 3. Indexing with minsearch

In [194]:
def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        documents.append({
            "source": chunk["source"],            
            "content": "\n".join(chunk["content"])  # convert list → string
        })
    return documents  
    

In [195]:
documents = prepare_documents(chunks)

In [197]:
from minsearch import Index
index = Index(text_fields=["content"])
index.fit(documents)

In [198]:
len(documents)

988

# Question 4. Searching and RAG

In [199]:
results = index.search("python function definition", num_results=5)

In [ ]:
results

# Question 5. Full RAG

In [221]:
import json
from openai import OpenAI

openai_client = OpenAI()

instructions = """" You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
<QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


In [230]:
def build_prompt(question,search_results):
    context = json.dumps(search_results, indent =2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text, response.usage.input_tokens, response.usage.output_tokens

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer



In [231]:
rag("python function definition")

('To define a function in Python, you use the `def` keyword followed by the function name and parentheses that may include parameters. Here’s a standard outline:\n\n```python\ndef function_name(parameters):\n    # Function body\n    statements\n    return value  # Optional\n```\n\n### Example 1: Basic Function\nHere\'s a simple function that adds two numbers:\n\n```python\ndef add_numbers(a, b):\n    result = a + b\n    return result\n```\n\n### Example 2: Function with No Return\nIf a function does not return a value, it can still perform actions:\n\n```python\ndef greet(name):\n    print(f"Hello, {name}!")\n```\n\n### Key Points\n- **Function Name:** Should be descriptive of what the function does (e.g., `add_numbers`).\n- **Parameters:** Variables listed in parentheses, used for passing information to the function.\n- **Function Body:** Contains operations to be executed.\n- **Return Statement:** Ends the function and can send a value back to the caller.\n\n### Calling Functions\nOn

# Question 6. Structured vs Unstructured Output

In [234]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [242]:
def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed, response.usage.input_tokens, response.usage.output_tokens

In [243]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [249]:
answer = rag_structured('python function definition', RAGResponse)

In [252]:
answer

(RAGResponse(answer='## Python Function Definition\n\nIn Python, a function is defined using the `def` keyword, followed by the function name and parentheses `()`. Any parameters should be included within the parentheses. The function block starts with a colon `:` and contains the code that defines what the function does. Here’s a simple outline of a function definition:\n\n```python\ndef function_name(parameters):\n    # function body\n    return result  # optional\n```\n\n### Example:\nHere is an example of a function that calculates the square of a number:\n\n```python\ndef square(x):\n    return x ** 2\n```\n\n### Calling the Function:\nYou can call this function like so:\n\n```python\nresult = square(4)  # result will be 16\n``` \n\n### Important Points:\n- **Function Name**: Should be descriptive and follow naming conventions (usually lowercase with underscores).\n- **Parameters**: Optional inputs to the function that can be used within its body.\n- **Return Statement**: Used to 